# Normalización

Los canales del sensor tienen unidades y escalas muy distintas: la aceleración se mueve en unos cuantos g y la velocidad angular en cientos de °/s. Muchos algoritmos de aprendizaje automático comparan muestras midiendo distancias o ajustan pesos con gradientes, y en ambos casos un canal con números grandes domina sobre los demás solo por su escala. Normalizar es llevar todos los canales a una escala comparable.

En este notebook se comparan tres métodos, se ve cómo les afectan los valores atípicos y se discuten dos decisiones importantes: con qué datos se calculan los parámetros de la normalización y cuándo se normaliza.

## Parámetros

In [ ]:
CARPETA = "datos/demo"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

from wt9011 import cargar_grabaciones
from procesamiento import preprocesar

grabaciones = cargar_grabaciones(CARPETA)
for g in grabaciones:
    g["pre"] = preprocesar(g["df"], g["fs"])
    g["pre"]["etiqueta"] = g["etiqueta"]
    g["pre"]["archivo"] = g["archivo"]

ETIQUETAS = sorted(set(g["etiqueta"] for g in grabaciones))
COLORES = {e: f"C{i}" for i, e in enumerate(ETIQUETAS)}

# primera grabación de cada actividad
ejemplo = {}
for g in grabaciones:
    if g["etiqueta"] not in ejemplo:
        ejemplo[g["etiqueta"]] = g

# todas las muestras de todas las grabaciones en una sola tabla
todas = pd.concat([g["pre"] for g in grabaciones], ignore_index=True)
print(f"{len(grabaciones)} grabaciones, {len(todas)} muestras")

## El problema de las escalas

Media, desviación estándar y rango de cada canal, con todas las grabaciones juntas:

In [ ]:
CANALES = ["ax", "ay", "az", "gx", "gy", "gz", "cuerpo_mag", "g_mag"]

todas[CANALES].describe().T[["mean", "std", "min", "max"]].round(2)

La desviación estándar de la velocidad angular es cientos de veces mayor que la de la aceleración. Para ver qué consecuencia tiene, se toman dos muestras de actividades distintas y se calcula la distancia euclidiana entre ellas, que es lo que usan métodos como k vecinos más cercanos. La tabla muestra qué porcentaje de esa distancia aporta cada canal.

In [ ]:
a = todas[todas["etiqueta"] == ETIQUETAS[0]].iloc[100]
b = todas[todas["etiqueta"] == ETIQUETAS[-1]].iloc[100]

diferencia2 = (a[CANALES] - b[CANALES]).astype(float) ** 2
print(f"Muestra de '{a['etiqueta']}' contra muestra de '{b['etiqueta']}'")
print(f"Distancia euclidiana: {np.sqrt(diferencia2.sum()):.1f}\n")
(100 * diferencia2 / diferencia2.sum()).round(2).rename("aporte a la distancia [%]")

Prácticamente toda la distancia viene del giroscopio. Para ese modelo la aceleración casi no existe, aunque contenga información útil para distinguir las actividades.

## Tres métodos

| Método | Fórmula | Resultado |
|---|---|---|
| z-score (estandarización) | (x − media) / desviación estándar | media 0 y desviación estándar 1 |
| min-max | (x − mín) / (máx − mín) | valores entre 0 y 1 |
| robusto | (x − mediana) / rango intercuartílico | mediana 0; la mitad central de los datos ocupa un ancho de 1 |

El rango intercuartílico es la distancia entre el percentil 25 y el 75. Ninguno de los tres cambia la forma de la señal: solo la desplazan y cambian su escala.

Primero se programan a mano y se comparan con las versiones de scikit-learn, que son las que se usan en la práctica.

In [ ]:
def zscore(x):
    return (x - x.mean()) / x.std()

def minmax(x):
    return (x - x.min()) / (x.max() - x.min())

def robusto(x):
    q25, q50, q75 = np.percentile(x, [25, 50, 75])
    return (x - q50) / (q75 - q25)

g = ejemplo[ETIQUETAS[0]]
t = g["pre"]["t"].to_numpy()
x = g["pre"]["g_mag"].to_numpy()
color = COLORES[g["etiqueta"]]

metodos = [("z-score", zscore, StandardScaler()),
           ("min-max", minmax, MinMaxScaler()),
           ("robusto", robusto, RobustScaler())]

for nombre, funcion, escalador in metodos:
    con_sklearn = escalador.fit_transform(x.reshape(-1, 1)).ravel()
    print(f"{nombre:<8} a mano igual a scikit-learn: {np.allclose(funcion(x), con_sklearn)}")

fig, ejes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
ejes[0].plot(t, x, lw=0.8, color=color)
ejes[0].set_ylabel("original\n|ω| [°/s]")
for ax, (nombre, funcion, _) in zip(ejes[1:], metodos):
    ax.plot(t, funcion(x), lw=0.8, color=color)
    ax.set_ylabel(nombre)
for ax in ejes:
    ax.grid(alpha=0.3)
ejes[0].set_title(g["archivo"])
ejes[-1].set_xlabel("Tiempo [s]")
plt.tight_layout()
plt.show()

## Efecto de un valor atípico

Un golpe al sensor o un error de transmisión produce un valor muy alejado del resto. Para ver cómo afecta a cada método se agrega un pico artificial de cinco veces el máximo de la señal y se mide qué ancho ocupa el 90 % central de los datos después de normalizar, con y sin el pico.

In [ ]:
con_pico = x.copy()
con_pico[len(x) // 2] = 5 * x.max()

def ancho_central(y):
    p5, p95 = np.percentile(y, [5, 95])
    return p95 - p5

filas = []
for nombre, funcion, _ in metodos:
    sin = ancho_central(funcion(x))
    con = ancho_central(funcion(con_pico))
    filas.append({"método": nombre, "sin pico": round(sin, 3), "con pico": round(con, 3),
                  "cambio [%]": round(100 * (con - sin) / sin, 1)})
pd.DataFrame(filas)

Min-max es el más afectado: el máximo ahora es el pico, así que el resto de los datos queda comprimido en una franja pequeña cerca de 0. El z-score también cambia, porque el pico aumenta la desviación estándar. El robusto casi no cambia, porque la mediana y los percentiles 25 y 75 no dependen de los valores extremos.

Por eso min-max solo conviene cuando el rango de los datos es conocido y estable, el z-score es la opción por defecto y el robusto se usa cuando hay valores atípicos que no se pueden eliminar.

## Normalizar cada grabación o todo el conjunto

Una tentación es normalizar cada grabación con su propia media y desviación estándar. Aquí se compara eso con usar la media y la desviación estándar de todas las grabaciones juntas, para una grabación de cada actividad.

In [ ]:
CANAL = "cuerpo_mag"
media = todas[CANAL].mean()
desv = todas[CANAL].std()

fig, ejes = plt.subplots(len(ETIQUETAS), 2, figsize=(12, 2.2 * len(ETIQUETAS)), sharex=True, sharey="col")
for i, etiqueta in enumerate(ETIQUETAS):
    d = ejemplo[etiqueta]["pre"]
    ejes[i, 0].plot(d["t"], zscore(d[CANAL]), lw=0.8, color=COLORES[etiqueta])
    ejes[i, 1].plot(d["t"], (d[CANAL] - media) / desv, lw=0.8, color=COLORES[etiqueta])
    ejes[i, 0].set_ylabel(etiqueta)

for ax in ejes.flat:
    ax.grid(alpha=0.3)
ejes[0, 0].set_title("z-score con la media y desviación de cada grabación")
ejes[0, 1].set_title("z-score con la media y desviación de todas")
ejes[-1, 0].set_xlabel("Tiempo [s]")
ejes[-1, 1].set_xlabel("Tiempo [s]")
plt.tight_layout()
plt.show()

Normalizadas una por una, todas las grabaciones terminan con la misma media y la misma desviación estándar, y el ruido de "quieto" queda amplificado hasta parecer tan intenso como una actividad con movimiento. Se pierde justamente la información que distingue a las actividades: cuánto se mueve el sensor. Con los parámetros del conjunto completo las diferencias de intensidad se conservan.

## Los parámetros se calculan solo con los datos de entrenamiento

Para evaluar un modelo se separan los datos en entrenamiento y prueba. Los datos de prueba representan datos nuevos que el modelo nunca vio, así que la media y la desviación estándar de la normalización se calculan solo con el entrenamiento y después se aplican, sin recalcular, a la prueba. Si se calcularan con todos los datos, información de la prueba se filtraría al entrenamiento y la evaluación saldría más optimista de lo que es. A esto se le llama fuga de datos (data leakage).

En scikit-learn esto corresponde a usar `fit` con el entrenamiento y `transform` con ambos conjuntos. Como ejemplo se deja la quinta grabación de cada actividad para prueba.

In [ ]:
es_prueba = todas["archivo"].str.endswith("_05.csv")
X_entrenamiento = todas.loc[~es_prueba, CANALES]
X_prueba = todas.loc[es_prueba, CANALES]

escalador = StandardScaler()
escalador.fit(X_entrenamiento)                # los parámetros salen solo del entrenamiento
Z_entrenamiento = escalador.transform(X_entrenamiento)
Z_prueba = escalador.transform(X_prueba)      # se aplican los mismos parámetros

print("Entrenamiento con media 0:       ", np.allclose(Z_entrenamiento.mean(axis=0), 0))
print("Entrenamiento con desviación 1:  ", np.allclose(Z_entrenamiento.std(axis=0), 1))

tabla = pd.DataFrame(index=CANALES)
tabla["media prueba"] = Z_prueba.mean(axis=0).round(2)
tabla["desv. prueba"] = Z_prueba.std(axis=0).round(2)
tabla

El entrenamiento queda con media 0 y desviación 1 exactas. La prueba no, y así debe ser: son datos que el escalador no vio. Cuanto más se alejan sus valores de 0 y 1, más distintas son las grabaciones de prueba de las de entrenamiento, por ejemplo porque se hicieron con más intensidad.

Dos notas finales. No todos los modelos necesitan normalización: los basados en árboles de decisión comparan cada variable contra un umbral por separado y no les afecta la escala. Y en este curso la normalización se aplicará al final, sobre la tabla de características de cada ventana y no sobre las señales; aquí se practicó con las señales para ver su efecto.

En `06_frecuencia` se analiza el contenido en frecuencia de las señales con la transformada de Fourier.